In [ ]:
%load_ext autoreload
%autoreload 2
from datetime import date
from athletics_performance import (
    Athlete,
    Event,
    Performance,
    ScoringTableResolver,
)

# Performance Scoring Workflow

This notebook demonstrates how to calculate the score (in points) of a performance based on:
- The **performance** result (time or distance)
- The **athlete's category** (determined by year of birth and competition season)
- The **athlete's sex** (M or F)

## The Scoring Pipeline

1. **Define an Athlete** with licence, name, and year of birth
2. **Create a Performance** with date, result, and event
3. **Determine the category** for the competition season (using `Athlete.category(yos)`)
4. **Select the scoring table** using `ScoringTableResolver.get(category)`
5. **Calculate the score** using `table.score(sex, event_id, result_value)`

## Example 1: MI Category (Minimes, 10-11 years old)

A young athlete competing in season 2026.

In [ ]:
# Create a young athlete (born 2014, will be 11-12 in season 2026)
sarah = Athlete(
    licence="4001",
    last_name="Dupont",
    first_name="Sarah",
    yob=2014,
    sex="F"
)

# Define an event
event_50m = Event(
    event_id="50m",
    name="50 mètres",
    measurement="time",
    unit="s"
)

# Create a performance: Sarah runs 50m in 7.20 seconds on May 10, 2026
perf_sarah = Performance(
    perf_id="SARAH_50M_001",
    date=date(2026, 5, 10),
    result_value=7.20,
    measurement="time",
    unit="s",
    athlete=sarah,
    event=event_50m,
    category_snapshot="MIF",
    club_id_snapshot="069069"
)

print(f"Athlete: {sarah.full_name}")
print(f"Date of birth: {sarah.yob}")
print(f"Sex: {sarah.sex}")
print()

# Determine category for season 2026
yos = 2026
category = sarah.category(yos)
print(f"Competition season: {yos}")
print(f"Athlete's age in season {yos}: {yos - sarah.yob}")
print(f"Category: {category}")
print()

# Get the scoring table for MI category
scoring_table = ScoringTableResolver.get(category)
print(f"Scoring table: {scoring_table.__class__.__name__}")
print(f"Applicable categories: {scoring_table.applicable_categories}")
print()

# Calculate the score
try:
    sex = sarah.sex  # "F"
    event_id = perf_sarah.event_id  # "50m"
    result = perf_sarah.result_value  # 7.20
    
    points = scoring_table.score(sex, event_id, result)
    print(f"Performance: {result}s in {event_id}")
    print(f"Points earned: {points} pts")
except Exception as e:
    print(f"Error calculating score: {e}")

## Example 2: BE Category (Benjamin, 14-15 years old)

In [ ]:
# Create a Benjamin athlete (born 2011, will be 14-15 in season 2026)
luc = Athlete(
    licence="4002",
    last_name="Bernard",
    first_name="Luc",
    yob=2011,
    sex="M"
)

event_800m = Event(
    event_id="800m",
    name="800 mètres",
    measurement="time",
    unit="s"
)

# Luc runs 800m in 2 minutes 10 seconds = 130 seconds
perf_luc = Performance(
    perf_id="LUC_800M_001",
    date=date(2026, 6, 15),
    result_value=130.0,
    measurement="time",
    unit="s",
    athlete=luc,
    event=event_800m,
    category_snapshot="BEM",
    club_id_snapshot="069106"
)

print(f"Athlete: {luc.full_name}")
print(f"Sex: {luc.sex}")
print()

category = luc.category(2026)
print(f"Category for season 2026: {category}")
print()

scoring_table = ScoringTableResolver.get(category)
print(f"Scoring table: {scoring_table.__class__.__name__}")
print()

try:
    points = scoring_table.score(luc.sex, perf_luc.event_id, perf_luc.result_value)
    minutes = int(perf_luc.result_value // 60)
    seconds = perf_luc.result_value % 60
    print(f"Performance: {minutes}'{seconds:.0f}\" in {perf_luc.event_id}")
    print(f"Points earned: {points} pts")
except Exception as e:
    print(f"Error calculating score: {e}")

## Example 3: Senior Category (SE) - World Athletics Scoring Table

In [ ]:
# Create a senior athlete (born 1990, will be 35-36 in season 2026)
# Actually let's use someone 23-34 to stay in SE category
marc = Athlete(
    licence="4003",
    last_name="Leclerc",
    first_name="Marc",
    yob=2000,
    sex="M"
)

event_100m = Event(
    event_id="100m",
    name="100 mètres",
    measurement="time",
    unit="s"
)

# Marc runs 100m in 10.50 seconds
perf_marc = Performance(
    perf_id="MARC_100M_001",
    date=date(2026, 7, 1),
    result_value=10.50,
    measurement="time",
    unit="s",
    athlete=marc,
    event=event_100m,
    category_snapshot="SEM",
    club_id_snapshot="069069"
)

print(f"Athlete: {marc.full_name}")
print(f"Sex: {marc.sex}")
print()

category = marc.category(2026)
print(f"Category for season 2026: {category}")
print()

scoring_table = ScoringTableResolver.get(category)
print(f"Scoring table: {scoring_table.__class__.__name__}")
print(f"Applicable categories: {scoring_table.applicable_categories}")
print()

try:
    points = scoring_table.score(marc.sex, perf_marc.event_id, perf_marc.result_value)
    print(f"Performance: {perf_marc.result_value}s in {perf_marc.event_id}")
    print(f"World Athletics Points: {points} pts")
except Exception as e:
    print(f"Error calculating score: {e}")

## Example 4: Multiple Performances in a Session

Calculate scores for multiple athletes and events in one go.

In [ ]:
# Create multiple athletes across different age categories
athletes_data = [
    {"name": "Emma", "licence": "5001", "yob": 2015, "sex": "F"},  # MI (10-11)
    {"name": "Thomas", "licence": "5002", "yob": 2011, "sex": "M"},  # BE (14-15)
    {"name": "Sophie", "licence": "5003", "yob": 2009, "sex": "F"},  # CA (16-17)
    {"name": "David", "licence": "5004", "yob": 2006, "sex": "M"},  # JU (18-19)
    {"name": "Claire", "licence": "5005", "yob": 2003, "sex": "F"},  # ES (20-22)
]

yos = 2026
athletes = [
    Athlete(
        licence=data["licence"],
        last_name="Test",
        first_name=data["name"],
        yob=data["yob"],
        sex=data["sex"]
    )
    for data in athletes_data
]

# Create performances for all athletes (all in 100m for easy comparison)
event_100m = Event(
    event_id="100m",
    name="100 mètres",
    measurement="time",
    unit="s"
)

performances = [
    Performance(
        perf_id=f"P{i:03d}",
        date=date(2026, 5, 10),
        result_value=14.50 + i * 0.5,  # 14.50s to 18.50s
        measurement="time",
        unit="s",
        athlete=athletes[i],
        event=event_100m,
    )
    for i in range(len(athletes))
]

print("=" * 80)
print(f"SCORING RESULTS - Season {yos}")
print("=" * 80)
print(f"{'Athlete':<15} {'YoB':>4} {'Age':>3} {'Category':>4} {'Event':>6} {'Result':>7} {'Points':>6}")
print("-" * 80)

for perf in performances:
    athlete = perf.athlete
    category = athlete.category(yos)
    age = yos - athlete.yob
    
    scoring_table = ScoringTableResolver.get(category)
    
    try:
        points = scoring_table.score(athlete.sex, perf.event_id, perf.result_value)
        result_str = f"{perf.result_value:.2f}s"
    except Exception as e:
        points = "N/A"
        result_str = f"{perf.result_value:.2f}s"
    
    print(
        f"{athlete.first_name:<15} {athlete.yob:>4} {age:>3} "
        f"{category:>4} {perf.event_id:>6} {result_str:>7} {str(points):>6}"
    )

print("=" * 80)

## Example 5: Long Jump (Field Event)

Demonstrate scoring with a field event, where higher distances are better.

In [ ]:
# Benjamin category - long jump
benjamin = Athlete(
    licence="6001",
    last_name="Moreau",
    first_name="Pierre",
    yob=2011,
    sex="M"
)

event_lj = Event(
    event_id="LJ",
    name="Saut en longueur",
    measurement="distance",
    unit="m"
)

# Pierre jumps 4.80m
perf_lj = Performance(
    perf_id="PIERRE_LJ_001",
    date=date(2026, 6, 1),
    result_value=4.80,
    measurement="distance",
    unit="m",
    athlete=benjamin,
    event=event_lj,
    category_snapshot="BEM"
)

print(f"Athlete: {benjamin.full_name}")
print(f"Category: {benjamin.category(2026)}")
print()

scoring_table = ScoringTableResolver.get(benjamin.category(2026))

try:
    points = scoring_table.score(benjamin.sex, perf_lj.event_id, perf_lj.result_value)
    print(f"Event: {perf_lj.event.name}")
    print(f"Distance: {perf_lj.result_value}m")
    print(f"Points: {points} pts")
except Exception as e:
    print(f"Error: {e}")

## Summary: The Complete Workflow

To calculate the score of a performance:

1. **Create an Athlete** with `licence`, `last_name`, `first_name`, `yob` (year of birth), and `sex`
2. **Create a Performance** with date, result_value, measurement type, unit, and references to the athlete and event
3. **Determine the category** using `athlete.category(yos)` for a given season
4. **Get the scoring table** using `ScoringTableResolver.get(category)` — automatically selects:
   - `MIYouthScoringTable` for MI (10-11 years)
   - `BEYouthScoringTable` for BE (14-15 years)
   - `WorldAthletics2025ScoringTable` for CA, JU, ES, SE, M0-M9 and above
5. **Calculate points** using `scoring_table.score(sex, event_id, result_value)`

### Available Methods on ScoringTable

- **`score(sex, event_id, performance_value)`** → points (integer)
- **`performance_for_points(sex, event_id, points)`** → performance value (for inverse lookup)
- **`applicable_categories`** → property listing which FFA categories use this table